# Towards a GIS system

In [ ]:
#| default_exp database

In [ ]:
import numpy as np
import sys
import os
import math
import random

#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
import json
from typing import List , Optional
from enum import Enum



In [ ]:
from dataclasses import dataclass
from typing import Optional

from datetime import datetime


In [ ]:
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx


In [ ]:
from importlib import resources
from pathlib import Path
import tempfile
import shutil


In [ ]:
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge
from HexMagic.styles import StyleCSS, SVGBuilder

In [ ]:
#!cat *.md

In [ ]:
#!cat ../llms.txt

#| export
## Helpers

In [ ]:
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

## Tables

In [ ]:
#| export


@dataclass
class TerrainWorld:
    """Metadata for a stored terrain."""
    id: int = None
    name: str = ""
    hex_radius: float = 25.0
    nrows: int = 0
    ncols: int = 0
    sea_level: float = 0.0
    elevation_delta: float = 90.0
    # Encoded extras (climate, geo bounds, etc)
    extras: str = ""
    created: int = 0
    modified: int = 0



In [ ]:
#| export
@dataclass  
class HexData:
    """Per-hex spatial data."""
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    grid_index: int = 0
    elevation: float = 0.0
    distance_from_coast: Optional[float] = None
    modified: int = 0  # Unix timestamp of last change


In [ ]:
#| export
@dataclass
class User:
    username: str
    email: str
    password: str
    created: int  # Unix timestamp
    sessionID: str
    activeWorld: int # a link into world_id for world 
    id: int = None

## Main Storage

In [ ]:
#| export
class GeoStorage:

    def __init__(self, custom_path=None):
        path = GeoStorage.get_db_path(custom_path)
        self.path = path
        self.createDB()

    def createDB(self):
        db = database(self.path)
        self.db = db

        # Create tables using dataclasses
        db.create(TerrainWorld, pk='id', if_not_exists=True, transform=True)
        db.create(HexData, pk='id', if_not_exists=True, transform=True)
        db.create(User, pk='id', if_not_exists=True, transform=True)

        # After creating tables
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_world ON hex_data(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_coords ON hex_data(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_grid ON hex_data(world_id, grid_index)")
        

        # Store table references using db.t notation
        self.users = db.t.user
        self.hexes = db.t.hex_data
        self.worlds = db.t.terrain_world

        
        
        # Populate templates (after tables exist)
        #self.populate_templates()


    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        
        # Try package data directory first
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            # Test if writable
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            # Fallback to user directory
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    
    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)

In [ ]:
#| export


#this might be something we don't export
@patch
def reset_database(self:GeoStorage):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)  # Convert string to Path
    
    if db_path.exists():
        db_path.unlink()

    self.createDB()



In [ ]:
#| export

@patch
def save_world(self: GeoStorage, terrain: Terrain, name: str = "") -> SaveResult:
    """Save terrain to database. Returns SaveResult(world_id, status, context)."""
    now = int(datetime.now().timestamp())
    
    grid = terrain.hexGrid
    
    # Build extras dict for non-field data
    extras = {}
    if terrain.geo:
        extras['geo'] = terrain.geo.encode() if hasattr(terrain.geo, 'encode') else str(terrain.geo)
    if terrain.climate:
        extras['climate'] = terrain.climate.encode() if hasattr(terrain.climate, 'encode') else str(terrain.climate)
    
    try:
        with self.db.conn:  # Transaction
            # Insert world metadata
            world_result = self.worlds.insert({
                'name': name or f"World_{now}",
                'hex_radius': grid.radius,
                'nrows': grid.nRows,
                'ncols': grid.nCols,
                'sea_level': terrain.seaLevel if isinstance(terrain.seaLevel, (int, float)) else 0.0,
                'elevation_delta': terrain.elevationDelta,
                'extras': json.dumps(extras),
                'created': now,
                'modified': now
            })
            
            world_id = world_result['id'] if isinstance(world_result, dict) else world_result
            
            # Insert hex data
            hex_records = []
            for idx, hex_obj in enumerate(grid.hexes):
                pos = grid.index_to_hexposition(idx)
                
                dist_coast = None
                if 'distance_to_ocean' in terrain.fields:
                    dist_coast = float(terrain.fields['distance_to_ocean'][idx])
                
                hex_records.append({
                    'world_id': world_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'grid_index': idx,
                    'elevation': float(terrain.elevations[idx]),
                    'distance_from_coast': dist_coast,
                    'modified': now
                })
            
            # Batch insert
            for record in hex_records:
                self.hexes.insert(record)
            
            return SaveResult(world_id, 'saved', f'{len(hex_records)} hexes')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def load_world(self: GeoStorage, world_id: int) -> LoadResult:
    """Load terrain from database. Returns LoadResult(terrain, status, context)."""
    try:
        # Get world metadata
        world = self.worlds[world_id]
        if not world:
            return LoadResult(None, 'not_found', f'World {world_id} not found')
        
        # Reconstruct bounds and grid
        radius = GeoStorage._get(world, 'hex_radius')
        nrows = GeoStorage._get(world, 'nrows')
        ncols = GeoStorage._get(world, 'ncols')
        
        # Calculate bounds from grid dimensions
        width = ncols * radius * 1.5 + radius
        height = nrows * radius * math.sqrt(3) + radius
        bounds = MapRect(MapCord(0, 0), MapSize(width, height))
        
        # Create terrain
        terrain = Terrain(bounds, radius=radius)
        terrain.hexGrid.nRows = nrows
        terrain.hexGrid.nCols = ncols
        terrain.hexGrid.adjustRadius(radius)  # Rebuilds hexes with correct dimensions
        terrain.elevations = np.zeros(len(terrain.hexGrid.hexes))  # <- Add this line
        terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')

        
        
        
        # Load hex data
        hex_rows = list(self.hexes.rows_where('world_id = ?', [world_id]))
        
        for row in hex_rows:
            idx = GeoStorage._get(row, 'grid_index')
            terrain.elevations[idx] = GeoStorage._get(row, 'elevation')
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(terrain.elevations))
                terrain.fields['distance_to_ocean'][idx] = dist
        
        # Restore extras
        extras_str = GeoStorage._get(world, 'extras')
        if extras_str:
            extras = json.loads(extras_str)
            # TODO: decode geo and climate from extras
        
        return LoadResult(terrain, 'loaded', f'{len(hex_rows)} hexes')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int) -> list:
    """Query all hexes within radius of a center position."""
    # Query with column names
    rows = list(self.hexes.rows_where("""
        world_id = ? AND q BETWEEN ? AND ? AND r BETWEEN ? AND ?
    """, [
        world_id,
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]))
    
    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)
    
    return result


In [ ]:
@patch
def extract_region(self: GeoStorage, world_id: int,
                   center_q: int, center_r: int, center_s: int,
                   rings: int, padding: int = 1) -> LoadResult:
    """Extract a centered region as a new Terrain.
    
    Returns LoadResult(terrain, status, context)
    """
    # Get world metadata for radius
    world = self.worlds[world_id]
    if not world:
        return LoadResult(None, 'not_found', f'World {world_id} not found')
    
    radius = GeoStorage._get(world, 'hex_radius')
    
    # Query hexes in region
    hex_rows = self.query_hexes_in_radius(world_id, center_q, center_r, center_s, rings)
    if not hex_rows:
        return LoadResult(None, 'empty', 'No hexes found in region')
    
    # Create centered grid
    total_rings = rings + padding
    grid = HexGrid.centered(radius=radius, rings=total_rings,style= StyleCSS("simple"))
    
    # Build terrain
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}
    terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')
    terrain.seaLevel = GeoStorage._get(world, 'sea_level') 
    terrain.geo = None
    terrain.climate = None
    terrain.path = ""
    terrain.colorLevels = None
    
    # Map DB rows to centered grid
    center_pos = HexPosition(center_q, center_r, center_s)
    filled = 0
    
    for row in hex_rows:
        # Position relative to query center
        q = GeoStorage._get(row, 'q') - center_q
        r = GeoStorage._get(row, 'r') - center_r
        s = GeoStorage._get(row, 's') - center_s
        rel_pos = HexPosition(q, r, s)
        
        # Map to centered grid (grid.middle is the center)
        new_idx = grid.hexposition_to_index(rel_pos, origin_index=grid.middle)
        if 0 <= new_idx < len(grid.hexes):
            terrain.elevations[new_idx] = GeoStorage._get(row, 'elevation')
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(grid.hexes))
                terrain.fields['distance_to_ocean'][new_idx] = dist
            
            filled += 1
    
    return LoadResult(terrain, 'extracted', f'{filled}/{len(hex_rows)} hexes mapped')


In [ ]:
@patch
def test_extract_region(self: GeoStorageDebugger):
    """Test extracting a centered region."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save full terrain
    save_result = self.server.save_world(terrain, name="Extract Test")
    assert save_result.status == 'saved'
    
    # Extract a region around origin
    result = self.server.extract_region(save_result.id, 0, 0, 0, rings=10)
    assert result.status == 'extracted', f"Extract failed: {result.context}"
    
    extracted = result.data
    print(f"✓ Extracted region: {result.context}")
    print(f"  Grid size: {len(extracted.hexGrid.hexes)} hexes")
    
    # Verify it's centered
    assert extracted.hexGrid.middle is not None
    print(f"  Center index: {extracted.hexGrid.middle}")


In [ ]:
def create_region_coord_mapper(center_q: int, center_r: int, center_s: int, 
                                extracted_grid: HexGrid) -> callable:
    """Create mapper from extracted region index to original world (q,r,s).
    
    Args:
        center_q, center_r, center_s: The center point used for extraction
        extracted_grid: The centered grid from extract_region
    
    Returns:
        Function: new_idx -> (q, r, s) tuple in original world coords
    """
    def mapper(new_idx: int) -> tuple[int, int, int]:
        # Get position relative to extracted grid's center
        rel_pos = extracted_grid.index_to_hexposition(new_idx, origin_index=extracted_grid.middle)
        # Add back the world center offset
        return (center_q + rel_pos.q, center_r + rel_pos.r, center_s + rel_pos.s)
    
    return mapper


In [ ]:
@patch
def create_region_index_mapper(self: GeoStorage, world_id: int,
                                center_q: int, center_r: int, center_s: int,
                                extracted_grid: HexGrid) -> callable:
    """Create mapper from extracted region index to original world grid_index.
    
    Args:
        world_id: The source world
        center_q, center_r, center_s: The center point used for extraction
        extracted_grid: The centered grid from extract_region
    
    Returns:
        Function: new_idx -> original_grid_index (or -1 if not in world)
    """
    # Build lookup dict from DB (q,r,s) -> grid_index
    hex_rows = list(self.hexes.rows_where('world_id = ?', [world_id]))
    coord_to_idx = {
        (GeoStorage._get(r, 'q'), GeoStorage._get(r, 'r'), GeoStorage._get(r, 's')): 
        GeoStorage._get(r, 'grid_index')
        for r in hex_rows
    }
    
    def mapper(new_idx: int) -> int:
        rel_pos = extracted_grid.index_to_hexposition(new_idx, origin_index=extracted_grid.middle)
        world_coords = (center_q + rel_pos.q, center_r + rel_pos.r, center_s + rel_pos.s)
        return coord_to_idx.get(world_coords, -1)
    
    return mapper


## Debugger

In [ ]:
#| export
class GeoStorageDebugger:
    """Test harness for HexServer with automatic cleanup."""
    
    def __init__(self, keep_on_error=False):
        """
        Args:
            keep_on_error: If True, preserve database when tests fail
        """
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = GeoStorage(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False
        
    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")
        
    def close(self):
        """Clean up test database."""
        self.server.db.close()
        
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()


In [ ]:
@patch
def test_save_load_terrain(self: GeoStorageDebugger):
    """Test saving and loading a terrain."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save
    save_result = self.server.save_world(terrain, name="Australia Test")
    assert save_result.status == 'saved', f"Save failed: {save_result.status} - {save_result.context}"
    print(f"✓ Saved world {save_result.id}: {save_result.context}")
    
    # Load
    load_result = self.server.load_world(save_result.id)
    assert load_result.status == 'loaded', f"Load failed: {load_result.status} - {load_result.context}"
    print(f"✓ Loaded world: {load_result.context}")
    
    # Verify
    loaded = load_result.data
    assert len(loaded.elevations) == len(terrain.elevations), "Elevation count mismatch"
    assert np.allclose(loaded.elevations, terrain.elevations), "Elevation values don't match"
    print(f"✓ Elevations match ({len(loaded.elevations)} hexes)")


In [ ]:
@patch
def test_query_radius(self: GeoStorageDebugger):
    """Test querying hexes by position and radius."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save
    save_result = self.server.save_world(terrain, name="Query Test")
    assert save_result.status == 'saved'
    
    # Query center region (q=0, r=0, s=0 with radius 5)
    hexes = self.server.query_hexes_in_radius(save_result.id, 0, 0, 0, radius=5)
    
    # Should get some hexes (exact count depends on map)
    print(f"✓ Found {len(hexes)} hexes within radius 5 of origin")
    assert len(hexes) > 0, "Expected some hexes near origin"


In [ ]:
@patch
def test_extract_region(self: GeoStorageDebugger):
    """Test extracting a centered region."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    # Save full terrain
    save_result = self.server.save_world(terrain, name="Extract Test")
    assert save_result.status == 'saved'
    
    # Extract a region around origin
    result = self.server.extract_region(save_result.id, 0, 0, 0, rings=10)
    assert result.status == 'extracted', f"Extract failed: {result.context}"
    
    extracted = result.data
    print(f"✓ Extracted region: {result.context}")
    print(f"  Grid size: {len(extracted.hexGrid.hexes)} hexes")
    
    # Verify it's centered
    assert extracted.hexGrid.middle is not None
    print(f"  Center index: {extracted.hexGrid.middle}")


In [ ]:
@patch
def test_region_mappers(self: GeoStorageDebugger):
    """Test that region mappers correctly translate indices."""
    td = TerraDemo()
    terrain = td.aussie_map()
    
    save_result = self.server.save_world(terrain, name="Mapper Test")
    assert save_result.status == 'saved'
    
    center_q, center_r, center_s = 5, -3, -2
    result = self.server.extract_region(save_result.id, center_q, center_r, center_s, rings=5)
    assert result.status == 'extracted'
    
    extracted = result.data
    grid = extracted.hexGrid
    
    # Test coord mapper
    coord_mapper = create_region_coord_mapper(center_q, center_r, center_s, grid)
    
    # Center of extracted grid should map back to query center
    center_coords = coord_mapper(grid.middle)
    assert center_coords == (center_q, center_r, center_s), f"Center mismatch: {center_coords}"
    print(f"✓ Coord mapper: center maps to {center_coords}")
    
    # Test index mapper  
    idx_mapper = self.server.create_region_index_mapper(
        save_result.id, center_q, center_r, center_s, grid
    )
    
    # Verify some mappings
    valid_count = sum(1 for i in range(len(grid.hexes)) if idx_mapper(i) >= 0)
    print(f"✓ Index mapper: {valid_count}/{len(grid.hexes)} indices map to original")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    dbg.test_save_load_terrain()
    dbg.test_query_radius()
    dbg.test_extract_region()
    dbg.test_region_mappers()


✓ Saved world 1: 4900 hexes


✓ Loaded world: 4900 hexes
✓ Elevations match (4900 hexes)


✓ Found 54 hexes within radius 5 of origin


✓ Extracted region: 181/181 hexes mapped
  Grid size: 529 hexes
  Center index: 264


✓ Coord mapper: center maps to (5, -3, -2)
✓ Index mapper: 143/169 indices map to original
✓ Test database cleaned up


In [ ]:
??HexRegion.crop_to_centered_grid


```python
@patch
def crop_to_centered_grid(self: HexRegion, style: StyleCSS = None, padding: int = 0) -> tuple[HexGrid, HexRegion, callable]:
    """Create a new grid cropped to fit this region, centered on centroid.

    Args:
        style: Style for the new grid's hexes (defaults to original grid's style)
        padding: Extra rings around the region

    Returns:
        (new_grid, new_region, index_mapper) tuple where:
        - new_grid: the cropped HexGrid
        - new_region: the region mapped to new_grid
        - index_mapper: function(new_idx) -> old_idx (or -1 if out of bounds)
    """
    if not self.hexes:
        return None, None, None

    grid = self.hexGrid
    style = style or grid.style

    # Use centroid as origin for shape
    origin_index = self.centroid_hex()

    # Convert all hexes to HexPositions relative to centroid
    shape_positions = [grid.index_to_hexposition(h, origin_index=origin_index) for h in self.hexes]

    # Find max distance from centroid to determine grid size
    max_dist = max(hp.distance(HexPosition.origin()) for hp in shape_positions)

    # Ensure odd number of rings for proper centering
    rings = (max_dist + padding) #//2 + 1
    if rings % 2 == 0:
        rings += 1

    # Create new centered grid
    new_grid = HexGrid.centered(radius=grid.radius, rings=rings, style=style)

    # Map shape positions to new grid (centered on middle)
    new_indices = set()
    for hp in shape_positions:
        new_idx = new_grid.hexposition_to_index(hp, origin_index=new_grid.middle)
        if new_idx >= 0:
            new_indices.add(new_idx)

    new_region = HexRegion(new_indices, new_grid)

    # Create index mapper: new_grid index -> original grid index
    def index_mapper(new_idx: int) -> int:
        """Map index from new grid to original grid. Returns -1 if out of bounds."""
        # Convert new index to HexPosition relative to new grid's middle
        hp = new_grid.index_to_hexposition(new_idx, origin_index=new_grid.middle)
        # Convert that position to index in original grid relative to original centroid
        return grid.hexposition_to_index(hp, origin_index=origin_index)

    return new_grid, new_region, index_mapper
```

**File:** `~/HexMagic/HexMagic/plot/region.py`

So we have made great progress on a great deal of the steps TerrainSchema.md. I want to put in a section about what we have built so far. are you able to cat to the end of the file. if not put it below and I can add it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I can't write to files directly. Here's the section to append to the end of `schemaMain.md`:

```markdown
---

## Implementation Progress

### Completed (2026-02-01)

#### Core Classes

**GeoStorage** - Main database manager
- `__init__(custom_path)` - Initialize with FastLite database
- `get_db_path()` - Package data or user fallback
- `reset_database()` - Delete and recreate (testing)
- `_get(obj, key)` - Dict/object attribute accessor

**Tables Created**
- `TerrainWorld` - World metadata (name, hex_radius, nrows, ncols, sea_level, elevation_delta, extras JSON)
- `HexData` - Per-hex spatial data (world_id, q, r, s, grid_index, elevation, distance_from_coast, modified timestamp)
- `User` - User accounts with activeWorld reference

**Indices**
- `idx_hex_world` - world_id lookup
- `idx_hex_coords` - (world_id, q, r, s) spatial queries
- `idx_hex_grid` - (world_id, grid_index) array-order queries

#### Save/Load Operations

**save_world(terrain, name)** → SaveResult
- Extracts grid parameters (radius, nrows, ncols)
- Stores geo/climate in extras JSON
- Batch inserts all HexData rows with elevation + distance_from_coast
- Returns SaveResult(world_id, status, context)

**load_world(world_id)** → LoadResult
- Reconstructs Terrain from TerrainWorld metadata
- Rebuilds grid with `adjustRadius()`
- Populates elevations and fields from HexData
- Returns LoadResult(terrain, status, context)

#### Spatial Queries

**query_hexes_in_radius(world_id, center_q, center_r, center_s, radius)** → list
- Bounding box SQL query followed by hex distance filter
- Returns list of HexData rows

**extract_region(world_id, center_q, center_r, center_s, rings, padding)** → LoadResult
- Creates centered HexGrid via `HexGrid.centered()`
- Maps DB rows to new grid using HexPosition arithmetic
- Returns LoadResult(terrain, status, context) with centered Terrain

#### Index Mappers

**create_region_coord_mapper(center_q, center_r, center_s, extracted_grid)** → callable
- Returns function: `new_idx → (q, r, s)` in world coordinates

**create_region_index_mapper(world_id, center_q, center_r, center_s, extracted_grid)** → callable
- Builds (q,r,s) → grid_index lookup from DB
- Returns function: `new_idx → original_grid_index` (or -1)

#### Testing

**GeoStorageDebugger** - Test harness with temp database
- Auto-cleanup unless `keep_on_error=True`
- `preserve_db(test_name)` - Copy to ~/.hexmagic/test_failures/

**Tests Passing**
- `test_save_load_terrain()` - Round-trip with elevation verification
- `test_query_radius()` - Spatial query returns expected hexes
- `test_extract_region()` - Centered grid extraction
- `test_region_mappers()` - Coordinate and index mapping

### Next Steps

1. **Downsampling** - `save_downsampled_world()` with ScaleTransform tracking
2. **Print rectangles** - `extract_rectangle()` for landscape/portrait crops
3. **Viewport persistence** - Save/load named viewports
4. **Additional fields** - Temperature, precipitation, watersheds
5. **Template system** - Pre-populate with TerraDemo maps
```

and how would you update llms.txt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's an updated `## Databases` section and related parts to replace in `llms.txt`:

```markdown
## Databases (2 Schemas)

### GeoStorage (HexMagic.database) - NEW
**Tables:** TerrainWorld, HexData, User

**Key Features:**
- Per-hex spatial storage (q, r, s coordinates + grid_index)
- Elevation and distance_from_coast fields with timestamps
- Spatial queries: `query_hexes_in_radius(world_id, q, r, s, radius)`
- Region extraction: `extract_region(world_id, q, r, s, rings)` → centered Terrain
- Index mappers for bidirectional coordinate translation

**Usage:**
```python
from HexMagic.database import GeoStorage, TerrainWorld, HexData

db = GeoStorage()
result = db.save_world(terrain, name="My World")
loaded = db.load_world(result.id)
region = db.extract_region(result.id, center_q=10, center_r=-5, center_s=-5, rings=20)
```

### HexServer (HexMagic.game.store) - Game State
**Tables:** User, Game, Autosave, Template

**Key Features:**
- GameBoard persistence with autosave history (last 10)
- Template library (7 pre-built maps)
- Session-based user management
- `start_from_template()` for quick game creation
```

And add to **Status**:

```markdown
**Completed:**
✅ Core terrain/geology with encode/decode
✅ Climate modeling (Köppen)
✅ Real-world data (7+ locations)
✅ SVG visualization
✅ Game mechanics basics
✅ GeoStorage database (save/load/extract)
✅ Per-hex spatial queries
✅ Region extraction with index mappers
✅ crop_to_centered_grid()
✅ downsample_climate()

**In Progress:**
🔄 ScaleTransform for downsampled worlds
🔄 Viewport persistence
🔄 Additional hex fields (temperature, precipitation)
```

so what I am wondering about is about handling changes. theorecial the value of a particular hex could change over time (think volcano) I know that there is a way to get the last value from sorted hexes and I am wondering if we need to put these into the querries we have built. This way for instance we could model how the terrain shifts over time.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking! You have two main approaches:

### Option 1: Append-only HexData (simple temporal queries)

Change inserts to never update, just append. Query latest with:

```python
@patch
def query_hexes_latest(self: GeoStorage, world_id: int) -> list:
    """Get most recent version of each hex."""
    return list(self.db.execute("""
        SELECT h.* FROM hex_data h
        INNER JOIN (
            SELECT world_id, q, r, s, MAX(modified) as max_mod
            FROM hex_data
            WHERE world_id = ?
            GROUP BY world_id, q, r, s
        ) latest ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """, [world_id]).fetchall())

@patch  
def query_hexes_at_time(self: GeoStorage, world_id: int, timestamp: int) -> list:
    """Get hex state at a specific point in time."""
    return list(self.db.execute("""
        SELECT h.* FROM hex_data h
        INNER JOIN (
            SELECT world_id, q, r, s, MAX(modified) as max_mod
            FROM hex_data
            WHERE world_id = ? AND modified <= ?
            GROUP BY world_id, q, r, s
        ) latest ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """, [world_id, timestamp]).fetchall())
```

### Option 2: Separate history table (cleaner current queries)

```python
@dataclass
class HexHistory:
    id: int = None
    hex_id: int = 0       # FK to hex_data.id
    world_id: int = 0
    elevation: float = 0.0
    distance_from_coast: Optional[float] = None
    timestamp: int = 0
    event_type: str = ""  # 'volcano', 'erosion', 'flood', etc.
```

### Recommendation

For your use case (geological events like volcanoes), I'd lean toward **Option 1** because:

1. Simpler schema - no FK management
2. Natural time-travel queries
3. Can replay history by querying successive timestamps
4. Already have `modified` field

The tradeoff is slightly more complex queries and larger table size. You'd want an additional index:

```python
db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")
```

Want me to update `load_world` and `extract_region` to use the temporal query pattern?

I definitely want option 1. I think as we add more columns or even possibly more tables this is the pattern we want to build up over time.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's update the core methods. First, add the index and helper queries:

```python
# Add to createDB after existing indices
db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")
```

```python
@patch
def _latest_hex_subquery(self: GeoStorage, world_id: int, as_of: int = None) -> str:
    """SQL subquery for latest hex state, optionally at a point in time."""
    time_clause = f"AND modified <= {as_of}" if as_of else ""
    return f"""
        SELECT world_id, q, r, s, MAX(modified) as max_mod
        FROM hex_data
        WHERE world_id = {world_id} {time_clause}
        GROUP BY world_id, q, r, s
    """

@patch
def query_hexes_latest(self: GeoStorage, world_id: int, as_of: int = None) -> list:
    """Get most recent version of each hex, optionally at a specific time."""
    subq = self._latest_hex_subquery(world_id, as_of)
    return list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """).fetchall())
```

Now update `load_world`:

```python
@patch
def load_world(self: GeoStorage, world_id: int, as_of: int = None) -> LoadResult:
    """Load terrain from database. 
    
    Args:
        world_id: World to load
        as_of: Optional Unix timestamp - load state at this point in time
    
    Returns LoadResult(terrain, status, context).
    """
    try:
        world = self.worlds[world_id]
        if not world:
            return LoadResult(None, 'not_found', f'World {world_id} not found')
        
        radius = GeoStorage._get(world, 'hex_radius')
        nrows = GeoStorage._get(world, 'nrows')
        ncols = GeoStorage._get(world, 'ncols')
        
        width = ncols * radius * 1.5 + radius
        height = nrows * radius * math.sqrt(3) + radius
        bounds = MapRect(MapCord(0, 0), MapSize(width, height))
        
        terrain = Terrain(bounds, radius=radius)
        terrain.hexGrid.nRows = nrows
        terrain.hexGrid.nCols = ncols
        terrain.hexGrid.adjustRadius(radius)
        terrain.elevations = np.zeros(len(terrain.hexGrid.hexes))
        terrain.elevationDelta = GeoStorage._get(world, 'elevation_delta')
        
        # Use temporal query
        hex_rows = self.query_hexes_latest(world_id, as_of=as_of)
        
        for row in hex_rows:
            idx = GeoStorage._get(row, 'grid_index')
            terrain.elevations[idx] = GeoStorage._get(row, 'elevation')
            
            dist = GeoStorage._get(row, 'distance_from_coast')
            if dist is not None:
                if 'distance_to_ocean' not in terrain.fields:
                    terrain.fields['distance_to_ocean'] = np.zeros(len(terrain.elevations))
                terrain.fields['distance_to_ocean'][idx] = dist
        
        extras_str = GeoStorage._get(world, 'extras')
        if extras_str:
            extras = json.loads(extras_str)
        
        time_info = f" as of {as_of}" if as_of else ""
        return LoadResult(terrain, 'loaded', f'{len(hex_rows)} hexes{time_info}')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))
```

And update `query_hexes_in_radius`:

```python
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    rows = list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]).fetchall())
    
    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)
    
    return result
```

Finally, add an update method that appends rather than modifies:

```python
@patch
def update_hex(self: GeoStorage, world_id: int, q: int, r: int, s: int,
               elevation: float = None, distance_from_coast: float = None,
               grid_index: int = None) -> SaveResult:
    """Append a new state for a hex (temporal update)."""
    now = int(datetime.now().timestamp())
    
    record = {
        'world_id': world_id,
        'q': q, 'r': r, 's': s,
        'modified': now
    }
    
    # Get previous state if we need to preserve fields
    prev = self.query_hexes_in_radius(world_id, q, r, s, radius=0)
    if prev:
        prev = prev[0]
        record['grid_index'] = grid_index if grid_index is not None else GeoStorage._get(prev, 'grid_index')
        record['elevation'] = elevation if elevation is not None else GeoStorage._get(prev, 'elevation')
        record['distance_from_coast'] = distance_from_coast if distance_from_coast is not None else GeoStorage._get(prev, 'distance_from_coast')
    else:
        record['grid_index'] = grid_index or 0
        record['elevation'] = elevation or 0.0
        record['distance_from_coast'] = distance_from_coast
    
    try:
        self.hexes.insert(record)
        return SaveResult(None, 'updated', f'hex ({q},{r},{s}) at {now}')
    except Exception as e:
        return SaveResult(None, 'error', str(e))
```

Update `extract_region` signature similarly - just add `as_of: int = None` param and pass it to `query_hexes_in_radius`.